# Train and test the gaze-only gate — 5-fold CV, AUC

Clones **[GazeVLM-HWSW-Codesign](https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign)**
and runs `src/foldtrain` on the fold CSVs already in Drive.

```bash
python -m src.foldtrain.train --folds_dir .../folds --out_dir runs/gate   # 5 folds
python -m src.foldtrain.infer --folds_dir .../folds --ckpt_dir runs/gate  # test once
```

## The task

Predict the gate decision **from the gaze signal alone** — no frame features, no DINOv2.
That is the constraint the deployed gate operates under: a model that needed the pixels to
decide whether to look at the pixels would be pointless.

| `--target` | Classes | AUC |
|---|---|---|
| `gate` | `SEND` / `DISCARD` | on P(SEND) |
| `quad` | `TRANSITION` / `PURSUIT` / `REFIXATION` / `STABLE` | macro one-vs-rest |

## About the T4

Set the runtime to a **T4 GPU**, but do not expect it to be the bottleneck. The gate never
opens a `.npz`, so the whole dataset is ~1.4 MB of floats that lives on the GPU for the
entire run — no DataLoader, no workers, no host-to-device traffic per step. The model is
~300 k parameters over a 9-step sequence.

The consequence: **an epoch takes milliseconds and all five folds finish in a couple of
minutes.** Batch 512 and AMP are set because they are free, not because they rescue
anything. The real runtime lever is early stopping.

It will also run fine on CPU, just slower.

## Order

1. Clone + mount
2. Check the fold files are there and consistent
3. **Train** 5 folds, watch the curves
4. **Shuffle control** — must land at AUC ≈ 0.5, or nothing else here is credible
5. **Test**, once, with the 5-model ensemble

## 1 — Clone, install, check the GPU

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -1

import os, json, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

print("\ntorch", torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU  {p.name}   {p.total_memory/1e9:.1f} GB   capability {p.major}.{p.minor}")
    print(f"     tensor cores: {'yes' if p.major >= 7 else 'no'} -> AMP "
          f"{'helps' if p.major >= 7 else 'will not help'}")
else:
    print("no GPU -- Runtime > Change runtime type > T4 GPU. It will still run on CPU.")

## 2 — Mount Drive and check the fold files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/GazeVLM"
FOLDS     = os.path.join(DRIVE_DIR, "folds")
RUNS      = "/content/runs/gate"

need = [f"train_val_fold{k}.csv" for k in range(1, 6)] + ["test.csv"]
missing = [f for f in need if not os.path.exists(os.path.join(FOLDS, f))]
assert not missing, (f"missing in {FOLDS}: {missing}\n"
                     f"Run notebooks/colab_label_quadrants.ipynb first.")

print(f"{FOLDS}")
for f in need:
    p = os.path.join(FOLDS, f)
    print(f"   {f:<24} {os.path.getsize(p)/1e6:6.1f} MB")

# sanity: the label columns must exist, and test must share no video with any fold
te = pd.read_csv(os.path.join(FOLDS, "test.csv"), usecols=["sequence", "quad", "gate"])
f1 = pd.read_csv(os.path.join(FOLDS, "train_val_fold1.csv"),
                 usecols=["sequence", "split", "quad", "gate"])
leak = set(te.sequence) & set(f1.sequence)
print(f"\ntest {len(te):,} rows / {te.sequence.nunique()} videos")
print(f"fold {len(f1):,} rows / {f1.sequence.nunique()} videos")
print(f"videos shared between test and fold 1: {len(leak)}   "
      f"{'[PASS]' if not leak else '[FAIL] ' + str(sorted(leak)[:5])}")

print("\nlabel balance:")
display(pd.DataFrame({
    "test":  te.quad.value_counts(normalize=True).mul(100).round(1),
    "fold1": f1.quad.value_counts(normalize=True).mul(100).round(1),
}).rename(index={0:"TRANSITION",1:"PURSUIT",2:"REFIXATION",3:"STABLE"}).fillna(0))

## 3 — Train, 5 folds

Defaults: batch 512, AdamW at 3e-4 with a OneCycle schedule, AMP on, class weights on,
early stopping when validation AUC has not improved for 20 epochs.

Class weights matter here: with median thresholds the quadrants come out skewed, and an
unweighted loss will happily never predict `REFIXATION` — the one class the two-threshold
design exists to capture.

Switch `TARGET` to `quad` for the 4-way problem, or `RATES_COL` to
`gaze_vec3d_rates_window` for the sphere-exact speed signal.

In [ ]:
TARGET    = "gate"                  # "gate" (binary) or "quad" (4 classes)
RATES_COL = "gaze_rates_window"     # or "gaze_vec3d_rates_window"
EPOCHS    = 200                     # a cap; early stopping usually fires well before
PATIENCE  = 20
BATCH     = 512
LR        = 3e-4

t0 = time.time()
!python -m src.foldtrain.train \
    --folds_dir "{FOLDS}" \
    --out_dir   "{RUNS}" \
    --target    {TARGET} \
    --rates_col {RATES_COL} \
    --epochs    {EPOCHS} \
    --patience  {PATIENCE} \
    --batch_size {BATCH} \
    --lr        {LR}

print(f"\ntotal {(time.time()-t0)/60:.1f} min")

### Learning curves and the fold spread

In [ ]:
res = json.load(open(os.path.join(RUNS, "cv_results.json")))
H = res["history"]

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
for k, h in H.items():
    ep = [r["epoch"] for r in h]
    ax[0].plot(ep, [r["train_loss"] for r in h], lw=1.2, label=k)
    ax[1].plot(ep, [r["auc"] for r in h], lw=1.5, label=k)
    ax[2].plot(ep, [r["bal_acc"] for r in h], lw=1.2, label=k)

ax[0].set_title("training loss"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=8)
ax[1].axhline(0.5, ls="--", c="k", lw=1, label="chance")
ax[1].set_title("validation AUC", fontweight="bold"); ax[1].set_xlabel("epoch")
ax[1].set_ylabel("AUC"); ax[1].legend(fontsize=8)
ax[2].set_title("validation balanced accuracy"); ax[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()

# `last` = mean over the final epochs. `best` = max over ALL epochs, which is biased
# upward and is NOT what to report -- on a shuffled control it still reaches high values.
aucs  = np.array([f["last"]["auc"] for f in res["folds"]])
bests = np.array([f["best"]["auc"] for f in res["folds"]])
tab = pd.DataFrame([{"AUC(last5)": f["last"]["auc"], "bal_acc": f["last"]["bal_acc"],
                     "AUC(best_ep)": f["best"]["auc"], "best_ep": f["best"]["epoch"]}
                    for f in res["folds"]])
tab.index = [f"fold {i+1}" for i in range(len(tab))]
display(tab.round(4))

print(f"CV AUC (last5)  {aucs.mean():.4f} +/- {aucs.std():.4f}   "
      f"(range {aucs.min():.4f} .. {aucs.max():.4f})   <- report this")
print(f"CV AUC (best)   {bests.mean():.4f}   <- biased upward by "
      f"{bests.mean()-aucs.mean():+.4f}, do not quote")
if aucs.std() > 0.05:
    print("\n  Wide spread: most of that headline number is which videos happened to land")
    print("  in validation, not the model. Report mean AND spread, never the best fold.")
if aucs.mean() < 0.55:
    print("\n  Near chance. Check the control below lands in the same place -- if it does")
    print("  not, the evaluation is leaking rather than the signal being absent.")

## 4 — The shuffle control

Identical settings, one change: **training labels are permuted.** The relationship the
model is supposed to learn no longer exists.

**This run must land at AUC ≈ 0.5.** If it does not, something is leaking — and every
number above becomes uninterpretable, however good it looked. Run it before believing the
real result, not after.

In [ ]:
RUNS_CTRL = "/content/runs/gate_control"

!python -m src.foldtrain.train \
    --folds_dir "{FOLDS}" --out_dir "{RUNS_CTRL}" \
    --target {TARGET} --rates_col {RATES_COL} \
    --epochs {EPOCHS} --patience {PATIENCE} --batch_size {BATCH} --lr {LR} \
    --shuffle_control 2>&1 | tail -18

ctrl = json.load(open(os.path.join(RUNS_CTRL, "cv_results.json")))
ca = np.array([f["last"]["auc"] for f in ctrl["folds"]])
ra = np.array([f["last"]["auc"] for f in res["folds"]])
cb = np.array([f["best"]["auc"] for f in ctrl["folds"]])

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(np.arange(5) - .19, ra, .38, label=f"real  {ra.mean():.3f}")
ax.bar(np.arange(5) + .19, ca, .38, label=f"shuffled  {ca.mean():.3f}", color="tab:red")
ax.axhline(0.5, ls="--", c="k", lw=1, label="chance")
ax.set_xticks(range(5)); ax.set_xticklabels([f"fold {i+1}" for i in range(5)])
ax.set_ylabel("validation AUC"); ax.set_ylim(0.35, 1.0); ax.legend()
ax.set_title("real vs shuffled labels (last-5 mean AUC)", fontweight="bold")
plt.tight_layout(); plt.show()

print(f"real     AUC {ra.mean():.4f} +/- {ra.std():.4f}")
print(f"shuffled AUC {ca.mean():.4f} +/- {ca.std():.4f}")
print(f"   (the shuffled run's BEST-epoch AUC is {cb.mean():.4f} -- that is the number a")
print(f"    max-over-epochs metric would have reported for data with no signal at all)")
gap = ra.mean() - ca.mean()
print(f"gap      {gap:+.4f}")

if abs(ca.mean() - 0.5) > 0.08:
    print("\n  !! THE CONTROL DID NOT LAND AT CHANCE. Something leaks -- suspect the")
    print("     split (a video in both train and val) or a label-derived input.")
    print("     Do not report the real number until this is resolved.")
elif gap < 0.05:
    print("\n  Real and shuffled are indistinguishable: gaze is not separating these")
    print("  classes at this scale. That is a clean negative, not a bug.")
else:
    print("\n  Control is at chance and the real run is clearly above it -- the signal")
    print("  is real. Now the test set is worth spending.")

## 5 — Test, once

The five fold models are averaged in probability space. Each was trained on a different
4/5 of the same pool, so averaging is the natural way to use all of them; per-model scores
print alongside so the ensemble gain is visible rather than assumed.

**Read this once.** Every extra look spends a little of the only unbiased estimate the
project has.

In [ ]:
PRED_CSV = "/content/runs/gate/test_preds.csv"

!python -m src.foldtrain.infer \
    --folds_dir "{FOLDS}" --ckpt_dir "{RUNS}" --out_csv "{PRED_CSV}"

### The test result, in one picture

In [ ]:
from src.foldtrain.metrics import roc_auc

pr = pd.read_csv(PRED_CSV)
pcols = [c for c in pr.columns if c.startswith("p_")]
names = [c[2:] for c in pcols]
proba = pr[pcols].to_numpy()
truth = np.array([names.index(t) for t in pr["truth"]])

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))

# ROC, one curve per class (one only, for the binary target)
for c, nm in enumerate(names):
    if len(names) == 2 and c == 0:
        continue
    y = (truth == c).astype(int); s = proba[:, c]
    o = np.argsort(-s)
    tp = np.cumsum(y[o]); fp = np.cumsum(1 - y[o])
    if tp[-1] == 0 or fp[-1] == 0:
        continue
    ax[0].plot(fp / fp[-1], tp / tp[-1], lw=2,
               label=f"{nm}  AUC {roc_auc(y, s):.3f}")
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate")
ax[0].set_title("ROC on the held-out test set", fontweight="bold"); ax[0].legend(fontsize=9)

cm = pd.crosstab(pr["truth"], pr["pred"]).reindex(index=names, columns=names, fill_value=0)
im = ax[1].imshow(cm.to_numpy(), cmap="Blues")
ax[1].set_xticks(range(len(names))); ax[1].set_xticklabels(names, rotation=30, ha="right")
ax[1].set_yticks(range(len(names))); ax[1].set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        v = cm.to_numpy()[i, j]
        ax[1].text(j, i, f"{v:,}", ha="center", va="center",
                   color="white" if v > cm.to_numpy().max() / 2 else "black", fontsize=9)
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("truth"); ax[1].set_title("confusion")

per = []
for c, nm in enumerate(names):
    m = truth == c
    per.append(dict(cls=nm, n=int(m.sum()),
                    recall=float((proba[m].argmax(1) == c).mean()) if m.any() else np.nan,
                    auc=roc_auc(truth == c, proba[:, c])))
pdf = pd.DataFrame(per)
ax[2].barh(pdf.cls, pdf.auc); ax[2].axvline(0.5, ls="--", c="k", lw=1)
ax[2].set_xlim(0, 1); ax[2].set_xlabel("one-vs-rest AUC"); ax[2].set_title("per class")
plt.tight_layout(); plt.show()

display(pdf.round(4))
print(f"\noverall test accuracy: {pr.correct.mean():.4f}")
print("\nA class with high AUC but low recall is being ranked well and thresholded badly")
print("-- that is fixable by moving the decision threshold, unlike a low AUC.")

---

## What each number means

| | |
|---|---|
| **CV AUC mean** | how well gaze separates the classes on unseen videos |
| **CV AUC spread** | how much of that is luck of the split. Wide spread → report both |
| **Control AUC** | must be ≈ 0.5. Anything higher means a leak, and invalidates the rest |
| **Test AUC** | the single unbiased number. Quote this one |
| **FALSE SKIP** | frames the gate dropped that the oracle would have kept — **content lost** |
| **FALSE SEND** | compute wasted, nothing lost. Much the cheaper error |

The two gate errors are not symmetric, which is why `infer.py` reports them separately
rather than folding both into accuracy.

## If AUC is near 0.5

Check in this order, cheapest first:

1. **Did the control also sit at 0.5?** If it sat higher, fix the leak before anything else.
2. **Is `REFIXATION` present at all?** A 1% class cannot be learned or measured. Re-run the
   labelling notebook with a lower `FRAME_PCT` and higher `GAZE_PCT`.
3. **Try `--rates_col gaze_vec3d_rates_window`** — sphere-exact, where `ω_mag` over-reads
   by 1/cos(pitch).
4. **Try `--target quad`** — the binary gate merges three quite different situations into
   `SEND`, which may be harder than separating all four.
5. **Warm-start** with `--init_from runs/loss2/best.pt` if a Loss-2 checkpoint exists.

## Things this notebook deliberately does not do

- **Tune on test.** Thresholds, architecture and stopping all use validation folds only.
- **Report the best fold.** The mean and the spread are the result; the best fold is noise.
- **Feed the model anything derived from the labels.** `FoldData` reads only the rates
  column, so `quad`, `gate` and both similarities cannot leak in structurally.